In [31]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

In [15]:
titanic_df = sns.load_dataset("titanic")
print(titanic_df.head(5))
print(titanic_df.shape)

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
(891, 15)


In [16]:
# checking missing values
print(titanic_df.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [17]:
#keeping useful columns only
titanic_df = titanic_df[[
    "survived",
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]]

In [18]:
# handling missing values
titanic_df["age"] = titanic_df["age"].fillna(titanic_df["age"].median())
titanic_df["embarked"] = titanic_df["embarked"].fillna(titanic_df["embarked"].mode()[0])

In [19]:
print(titanic_df.isnull().sum())

survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [20]:
print(titanic_df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked'],
      dtype='str')


In [21]:
# converting the categorical values into dummy variables as 0 and 1. 
# Since sex and embarked contains categorical values so converting them into
# dummy variables while keeping the rest of the dataframe as it is.
titanic_df = pd.get_dummies(titanic_df, columns=['sex', 'embarked'], drop_first=True)

In [24]:
print(titanic_df.head(3))

   survived  pclass   age  sibsp  parch     fare  sex_male  embarked_Q  \
0         0       3  22.0      1      0   7.2500      True       False   
1         1       1  38.0      1      0  71.2833     False       False   
2         1       3  26.0      0      0   7.9250     False       False   

   embarked_S  
0        True  
1       False  
2        True  


In [ ]:
# computing Pearson correlation coefficient for all columns pairs in dataset
# Pearson correlation tells:
"""
How strongly two variables moves together
1 -> strong positive relation
-1 -> strong negative relation
0 -> no relation
"""

correlations = titanic_df.corr().abs()

top_5_features = correlations["survived"].sort_values(ascending=False).iloc[1:6]

print(top_5_features)

sex_male      0.543351
pclass        0.338481
fare          0.257307
embarked_S    0.149683
parch         0.081629
Name: survived, dtype: float64


In [28]:
# features and target
X = titanic_df[top_5_features.index]
y = titanic_df['survived']

In [29]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=417)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.25, random_state=417)

In [ ]:
#normalize all the features in the training data by scaling their values to the range [0, 1].
scaler = MinMaxScaler()
X_trained_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [32]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_trained_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [33]:
val_accuracy = knn_model.score(X_val_scaled, y_val)
print(val_accuracy)

0.8212290502793296


In [34]:
# Tuning hyperparameter (K)

for k in [1, 3, 5, 7, 9]:
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_trained_scaled, y_train)
    val_accuracy = knn_model.score(X_val_scaled, y_val)

    print(k, val_accuracy)


1 0.7262569832402235
3 0.8212290502793296
5 0.8659217877094972
7 0.8659217877094972
9 0.8715083798882681


In [35]:
# since k = 9 is giving highest validation accuracy i.e. 0.87
#picking k = 9 to train the model

knn_final_model = KNeighborsClassifier(n_neighbors=9)
knn_final_model.fit(X_trained_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",9
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [36]:
# final evaluation on test set
test_accuracy = knn_final_model.score(X_test_scaled, y_test)
print(test_accuracy)

0.7696629213483146


In [ ]:
"""
CONCLUSION:
Since the validation accuracy is ~87% and
test accuracy is ~76% that means the model is
overfitting somewhere.
The possible reasons could be small dataset,
using only top 5 features from the dataset (limited features) etc.

The solution can be:
Increasing the dataset,
including more features
try better models such as logistic regression, decision tree etc.

The takeaway:
High validation accuracy does not guarantee high test accuracy.
"""